## Install Libraries

In [28]:
 !pip install -q sentence-transformers faiss-cpu gradio kagglehub python-docx transformers accelerate bitsandbytes cohere fitz tools pymupdf

## Imports

In [29]:
import os
#import json
import re
import torch
import numpy as np
#import pandas as pd
import docx
from tqdm import tqdm
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
 
# Hugging Face 
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
import cohere
import faiss
 
# Gradio & scraping
import gradio as gr
import requests
from bs4 import BeautifulSoup
 

## Importing HF

In [30]:
user_secrets = UserSecretsClient()
 
# HuggingFace login (unchanged)
hf_token = user_secrets.get_secret("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged into Hugging Face")
else:
    print("No HF token found. Base Qwen may still work.")
 
# Cohere client (new)
COHERE_API_KEY = user_secrets.get_secret("COHERE_API")
co = cohere.Client(COHERE_API_KEY)
print("✅ Cohere client initialized")
 

Logged into Hugging Face
✅ Cohere client initialized


## Model Loading

In [31]:
# 4-bit quantization config (saves memory)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading base Qwen 3B model...")
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
print("✅ Base model loaded")

Loading base Qwen 3B model...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✅ Base model loaded


### Testing model

In [32]:
def test_model_generation(prompt):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.7, do_sample=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=" * 50)
print("TEST: Basic conversation")
print("=" * 50)
result = test_model_generation("What is your role as a career coach?")
print(result)

print("\n" + "=" * 50)
print("TEST: CV completion")
print("=" * 50)
result2 = test_model_generation("Give me the perfect structure for a CV")
print(result2)

print("\n✅ Base model is working. Now proceed to job scraping and RAG.")

TEST: Basic conversation
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What is your role as a career coach?
assistant
As a career coach, my primary role is to help individuals identify their career goals and develop strategies to achieve them. I assist in enhancing personal and professional skills, helping people to make informed decisions about their careers, and providing guidance on job searching, networking, and further education opportunities.

Here are some key aspects of what I do:

1. **Career Assessment**: Understanding an individual's interests, values, skills, and personality traits to align their career path with their personal preferences and strengths.
2. **

TEST: CV completion
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Give me the perfect structure for a CV
assistant
Certainly! A well-structured resume (CV) is crucial for making a good first impression and highlighting your relevant skills and expe

##  Job Scraping

In [33]:
## Job Scraping — Full Code (With Cohere Semantic Filtering)
## Cohere embed-multilingual-v3.0 semantic filtering instead of hardcoded keywords

import re, html, os, urllib.parse, xml.etree.ElementTree as ET
import requests
import numpy as np
from typing import List, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed
try:
    from bs4 import BeautifulSoup
    _BS4 = True
except ImportError:
    _BS4 = False
    import warnings
    warnings.warn("beautifulsoup4 not found — pip install beautifulsoup4", stacklevel=1)

try:
    from tenacity import retry, stop_after_attempt, wait_fixed, retry_if_exception_type
    _TENACITY = True
except ImportError:
    _TENACITY = False

_H = {"User-Agent": "NLP-3AlInfo3/1.0", "Accept": "application/json, text/html, */*"}

In [34]:
# ── Secrets ────────────────────────────────────────────────────────────────────
user_secrets = UserSecretsClient()
RAPIDAPI_KEY   = user_secrets.get_secret("RAPIDAPI_KEY")
FINDWORK_KEY   = user_secrets.get_secret("FINDWORK_API_KEY") # register free: findwork.dev

In [35]:
# ── Helpers ────────────────────────────────────────────────────────────────────
def _clean(text: str) -> str:
    """Strip HTML tags and collapse whitespace."""
    if not text:
        return ""
    if _BS4:
        soup = BeautifulSoup(text, "html.parser")
        for tag in soup(["script", "style"]):
            tag.decompose()
        text = soup.get_text(separator=" ")
    else:
        text = re.sub(r"<[^>]+>", " ", html.unescape(text))
    return re.sub(r"\s+", " ", text).strip()


In [36]:
from typing import List, Dict
def filter_tech_jobs_semantic(jobs: List[Dict], threshold: float = 0.38) -> List[Dict]:
    """
    Filter jobs using Cohere's Multilingual Embeddings.
    Computes cosine similarity between the job description and a semantic tech anchor text.
    Processes in optimal batches of 96 to respect Cohere API limits.
    """
    if not jobs:
        return []

    tech_anchor = (
        "This is a technology, IT, software engineering, web development, coding, "
        "data science, AI/ML, DevOps, cybersecurity, tech product management, or UI/UX design job."
    )

    # Prepare job texts (title + top of description)
    job_texts = [f"Title: {j['title']}\nDescription: {j['description'][:500]}" for j in jobs]
    all_texts = [tech_anchor] + job_texts

    all_embeddings = []
    print(f"  [semantic] Embedding {len(all_texts)} items in batches of 96 using Cohere...")
    for i in range(0, len(all_texts), 96):
        batch = all_texts[i:i + 96]
        response = co.embed(
            texts=batch,
            model="embed-multilingual-v3.0",
            input_type="search_document",
            truncate="END",
        )
        all_embeddings.extend(response.embeddings)

    embeddings = np.array(all_embeddings, dtype="float32")

    # Extract normalized reference and job embeddings
    ref_emb = embeddings[0] / np.linalg.norm(embeddings[0])
    job_embs = embeddings[1:]

    valid_jobs = []
    for job, emb in zip(jobs, job_embs):
        norm_emb = emb / np.linalg.norm(emb)
        similarity = np.dot(ref_emb, norm_emb)
        
        # Keep job if similarity is above threshold
        if similarity >= threshold:
            job["semantic_score"] = float(similarity)
            valid_jobs.append(job)

    return valid_jobs

def _with_retry(fn):
    if not _TENACITY:
        return fn
    return retry(
        stop=stop_after_attempt(3),
        wait=wait_fixed(2),
        retry=retry_if_exception_type((requests.ConnectionError, requests.Timeout)),
        reraise=True,
    )(fn)

# ── Scrapers ───────────────────────────────────────────────────────────────────

@_with_retry
def _fetch_jsearch(query: str, location: str, n: int) -> List[Dict]:
    """JSearch via RapidAPI — pulls LinkedIn + Indeed results."""
    url = "https://jsearch.p.rapidapi.com/search"
    headers = {**_H, "X-RapidAPI-Key": RAPIDAPI_KEY, "X-RapidAPI-Host": "jsearch.p.rapidapi.com"}
    params = {
        "query": f"{query} {location}".strip(),
        "num_pages": 1,
        "page": 1,
        "results_per_page": min(n, 10),
    }
    data = requests.get(url, headers=headers, params=params, timeout=15).json()
    out = []
    for item in data.get("data", [])[:n]:
        out.append({
            "title":       item.get("job_title", "Unknown"),
            "company":     item.get("employer_name", "Unknown"),
            "location":    f"{item.get('job_city','')} {item.get('job_country','')}".strip() or "Remote",
            "description": item.get("job_description", "")[:2500],
            "source":      "JSearch (LinkedIn/Indeed)",
        })
    return out

@_with_retry
def _fetch_wwr(query: str, n: int) -> List[Dict]:
    """We Work Remotely RSS feed."""
    r = requests.get("https://weworkremotely.com/remote-jobs.rss", timeout=18, headers=_H)
    r.raise_for_status()
    root = ET.fromstring(r.content)
    qw = query.lower().split()
    out = []
    for item in root.iter("item"):
        t = item.find("title")
        d = item.find("description")
        if t is None:
            continue
        raw = html.unescape(t.text or "")
        company, title = raw.split(": ", 1) if ": " in raw else ("Unknown", raw)
        desc = _clean(d.text or "" if d is not None else "")
        if not any(w in f"{title} {desc}".lower() for w in qw):
            continue
        out.append({
            "title":       title.strip(),
            "company":     company.strip(),
            "location":    "Remote",
            "description": desc[:2500],
            "source":      "We Work Remotely",
        })
        if len(out) >= n:
            break
    return out

@_with_retry
def _fetch_arbeit_now(query: str, n: int) -> List[Dict]:
    """
    Arbeit Now — mostly European jobs.
    """
    url = f"https://www.arbeitnow.com/api/job-board-api?search={urllib.parse.quote_plus(query)}"
    resp = requests.get(url, timeout=12, headers=_H)
    resp.raise_for_status()
    data = resp.json().get("data", [])

    if not data:
        fallback_resp = requests.get(
            "https://www.arbeitnow.com/api/job-board-api", timeout=12, headers=_H
        )
        fallback_resp.raise_for_status()
        fallback = fallback_resp.json().get("data", [])
        qw = query.lower().split()
        data = [
            item for item in fallback
            if all(w in f"{item.get('title','')} {item.get('description','')}".lower() for w in qw)
            and any(w in item.get("title", "").lower() for w in qw)
        ]

    out = []
    for item in data[:n]:
        out.append({
            "title":       item.get("title", "Unknown"),
            "company":     item.get("company_name", "Unknown"),
            "location":    item.get("location", "Remote"),
            "description": _clean(item.get("description", ""))[:2500],
            "source":      "Arbeit Now",
        })
    return out


@_with_retry
def _fetch_remotive(query: str, n: int) -> List[Dict]:
    """Remotive — remote tech jobs only, no auth required."""
    url = "https://remotive.com/api/remote-jobs"
    params = {
        "category": "software-dev",
        "limit":    min(n, 100),
        "search":   query,
    }
    resp = requests.get(url, params=params, timeout=12, headers=_H)
    resp.raise_for_status()
    jobs = resp.json().get("jobs", [])
    out = []
    for item in jobs[:n]:
        out.append({
            "title":       item.get("title", "Unknown"),
            "company":     item.get("company_name", "Unknown"),
            "location":    item.get("candidate_required_location", "Remote"),
            "description": _clean(item.get("description", ""))[:2500],
            "source":      "Remotive",
        })
    return out


@_with_retry
def _fetch_jobicy(query: str, n: int) -> List[Dict]:
    """Jobicy — remote tech jobs, structured tags, no auth required."""
    url = "https://jobicy.com/api/v2/remote-jobs"
    params = {
        "count":    min(n, 50),
        "industry": "engineering",
        "tag":      query,
    }
    resp = requests.get(url, params=params, timeout=12, headers=_H)
    resp.raise_for_status()
    jobs = resp.json().get("jobs", [])
    out = []
    for item in jobs[:n]:
        out.append({
            "title":       item.get("jobTitle", "Unknown"),
            "company":     item.get("companyName", "Unknown"),
            "location":    item.get("jobGeo", "Remote"),
            "description": _clean(item.get("jobDescription", ""))[:2500],
            "source":      "Jobicy",
        })
    return out


@_with_retry
def _fetch_findwork(query: str, n: int) -> List[Dict]:
    """Findwork.dev — developer-only job board, very clean data."""
    if not FINDWORK_KEY:
        return []
    url = "https://findwork.dev/api/jobs/"
    params = {
        "search":   query,
        "remote":   "true",
        "order_by": "relevance",
    }
    headers = {**_H, "Authorization": f"Token {FINDWORK_KEY}"}
    resp = requests.get(url, params=params, headers=headers, timeout=12)
    resp.raise_for_status()
    results = resp.json().get("results", [])
    out = []
    for item in results[:n]:
        keywords = ", ".join(item.get("keywords", []))
        out.append({
            "title":       item.get("role", "Unknown"),
            "company":     item.get("name", "Unknown"),
            "location":    item.get("location", "Remote"),
"description": _clean(f"{item.get('text', '')} Required skills: {keywords}")[:2500],
            "source":      "Findwork",
        })
    return out

In [37]:
# ── Main Search Function ───────────────────────────────────────────────────────

def search_jobs(query: str, location: str = "", limit: int = 10, semantic_threshold: float = 0.38) -> List[Dict]:
    """
    Scrape jobs from multiple sources, deduplicate, filter semantically using Cohere, and return.
    """
    fetch_n = limit * 4
    pool = []
    sources = []

    # API key required
    if RAPIDAPI_KEY:
        sources.append(("JSearch (LinkedIn/Indeed)", lambda: _fetch_jsearch(query, location, fetch_n)))
    else:
        print("  [skip] JSearch — add RAPIDAPI_KEY to Kaggle Secrets")

    if FINDWORK_KEY:
        sources.append(("Findwork",   lambda: _fetch_findwork(query, fetch_n)))
    else:
        print("  [skip] Findwork — add FINDWORK_API_KEY to Kaggle Secrets")

    # No auth required
    sources += [
        ("Remotive",         lambda: _fetch_remotive(query, fetch_n)),
        ("Jobicy",           lambda: _fetch_jobicy(query, fetch_n)),
        ("We Work Remotely", lambda: _fetch_wwr(query, fetch_n)),
       # ("Arbeit Now",       lambda: _fetch_arbeit_now(query, fetch_n)),
    ]

    for name, fn in sources:
        try:
            jobs = fn()
            pool.extend(jobs)
            print(f"  [ok]   {name}: {len(jobs)} job(s)")
        except Exception as e:
            print(f"  [err]  {name}: {e}")
    with ThreadPoolExecutor(max_workers=6) as executor:
        future_to_name = {
            executor.submit(fn): name
            for name, fn in sources
        }

        for future in as_completed(future_to_name):
            name = future_to_name[future]

            try:
                jobs = future.result()
                pool.extend(jobs)
                print(f"  [ok] {name}: {len(jobs)} job(s)")

            except Exception as e:
                print(f"  [err] {name}: {e}")
                
    # Deduplication
    seen, unique = set(), []
    for job in pool:
    # Handle None values in title and company
        title = job.get("title") or ""
        company = job.get("company") or ""
        key = (title.lower().strip()[:40], company.lower().strip()[:40])
        if key not in seen:
            seen.add(key)
            unique.append(job)
            
    # Quality filters
    before = len(unique)
    unique = [j for j in unique if j.get("title", "Unknown") != "Unknown"]
    unique = [j for j in unique if len(j.get("description", "")) > 100]
    
    unique = filter_tech_jobs_semantic(
        unique,
        threshold=semantic_threshold
    )
    
    after = len(unique)

    print(f"\n  Raw pool: {len(pool)} | After dedup: {before} | After semantic filter: {after}")
    return unique

#### Jobs sracping call method

In [38]:
# 1. Define query and run search
SEARCH_QUERY = "python"
print(f"Running scraper and semantic filter for: '{SEARCH_QUERY}'...")

# Using default threshold of 0.38
scraped_jobs = search_jobs(
    query=SEARCH_QUERY, 
    location="", 
    limit=15, 
    semantic_threshold=0.45 
)

# 2. Print formatted results
print("\n" + "="*80)
print(f"SEMANTIC FILTER RESULTS (Total Kept: {len(scraped_jobs)})")
print("="*80)

if not scraped_jobs:
    print("No jobs matched your threshold! Try lowering 'semantic_threshold' slightly (e.g., to 0.35).")
else:
    for idx, job in enumerate(scraped_jobs, 1):
        score = job.get("semantic_score", 0.0)
        print(f"{idx:02d}. [{score:.3f}] | {job['title']} at {job['company']}")
        print(f"     Source: {job['source']} | Location: {job['location']}")
        print(f"     Snippet: {job['description']}...")
        print("-"*80)

Running scraper and semantic filter for: 'python'...
  [ok]   JSearch (LinkedIn/Indeed): 10 job(s)
  [ok]   Findwork: 60 job(s)
  [ok]   Remotive: 18 job(s)
  [ok]   Jobicy: 3 job(s)
  [ok]   We Work Remotely: 9 job(s)
  [ok] Jobicy: 3 job(s)
  [ok] Remotive: 18 job(s)
  [ok] Findwork: 60 job(s)
  [ok] We Work Remotely: 9 job(s)
  [ok] JSearch (LinkedIn/Indeed): 10 job(s)
  [semantic] Embedding 94 items in batches of 96 using Cohere...

  Raw pool: 200 | After dedup: 93 | After semantic filter: 66

SEMANTIC FILTER RESULTS (Total Kept: 66)
01. [0.477] | Python Software Engineer at Atem Corp
     Source: JSearch (LinkedIn/Indeed) | Location: Washington US
     Snippet: Software Engineer Python with Angular, TypeScript

US & Canada - Remote

6+ Months Con

The Business Systems Team's Main Responsibility Is To Build, Enhance, Support, And Innovate On a Variety Of Key Business Facing Applications That Power Black Book's Operations. Our Work Spans API Development, Data Processing, Web Applic

## Build FAISS Index of Job Descriptions

In [39]:
def cohere_embed(texts: List[str], input_type: str) -> np.ndarray:
    """
    Embed texts using Cohere multilingual model.
    input_type: 'search_document' for jobs, 'search_query' for CV query.
    Batched at 96 (Cohere API max). Token limit: 512 per text.
    """
    all_embeddings = []
    for i in range(0, len(texts), 96):
        batch = texts[i:i + 96]
        response = co.embed(
            texts=batch,
            model="embed-multilingual-v3.0",
            input_type=input_type,
            truncate="END",
        )
        all_embeddings.extend(response.embeddings)
    return np.array(all_embeddings, dtype="float32")


def build_job_embed_text(job: dict) -> str:
    """Enrich job text for embedding — keeps title/company at top within token budget."""
    desc = job["description"][:1800]   # ~490 tokens left after header
    return (
        f"Job Title: {job['title']}\n"
        f"Company:   {job['company']}\n"
        f"Location:  {job['location']}\n\n"
        f"{desc}"
    )


# Build FAISS index
job_descs = [build_job_embed_text(job) for job in scraped_jobs]
print(f"Embedding {len(job_descs)} jobs with Cohere embed-multilingual-v3.0...")
job_embs = cohere_embed(job_descs, input_type="search_document")

faiss.normalize_L2(job_embs)
index = faiss.IndexFlatIP(job_embs.shape[1])
index.add(job_embs)
print(f"✅ FAISS index built: {len(job_descs)} jobs, dim={job_embs.shape[1]}")

# Sanity check
#test_q = cohere_embed(["developer"], input_type="search_query")
#faiss.normalize_L2(test_q)
#scores, idxs = index.search(test_q, 3)
#print("\n🔍 Sanity check:")
#for s, i in zip(scores[0], idxs[0]):
#    print(f"  {s:.3f} — {scraped_jobs[i]['title']} at {scraped_jobs[i]['description'][:50]}")

Embedding 66 jobs with Cohere embed-multilingual-v3.0...
✅ FAISS index built: 66 jobs, dim=1024


In [40]:
print("Job Embeddings : ", job_embs)

Job Embeddings :  [[ 0.02909251  0.04050971 -0.03296947 ...  0.09628306 -0.03849491
  -0.02017854]
 [-0.0152551   0.01890107 -0.0288474  ...  0.06266796  0.01964857
  -0.03008306]
 [ 0.02722483  0.03818191 -0.01977768 ...  0.07215191 -0.01797693
   0.02545461]
 ...
 [ 0.04580714  0.03424091 -0.04342676 ...  0.06439245 -0.0265657
  -0.04211449]
 [ 0.00667349  0.03888863 -0.0582719  ...  0.08748416 -0.08260018
   0.00041948]
 [ 0.01618954  0.01200864 -0.03933707 ...  0.10632301 -0.05694568
  -0.02151485]]


## RAG Retrieval Function

In [41]:
import pymupdf as fitz   # or: from pymupdf import open as fitz_open

def extract_text_from_file(file_path):
    """
    Extract text from DOCX or PDF files.
    Returns extracted text as string, or error message on failure.
    """
    ext = os.path.splitext(file_path)[1].lower()
    try:
        if ext == '.docx':
            doc = docx.Document(file_path)
            full_text = [para.text for para in doc.paragraphs if para.text.strip()]
            return '\n'.join(full_text).strip()
        elif ext == '.pdf':
            doc = fitz.open(file_path)
            text = ""
            for page in doc:
                text += page.get_text()
            doc.close()
            return text.strip()
        else:
            return f"Unsupported file type: {ext}. Please upload DOCX or PDF."
    except Exception as e:
        return f"Error reading file: {e}"

cv_text = extract_text_from_file("/kaggle/input/datasets/medazizbentourkia/gliner-training-data/CV_2023-06-14_Med Aziz_BenTourkia.pdf")

In [42]:
import re
import unicodedata

def clean_cv_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)                              # fix â€™, â¨, ð©
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', text)    # control chars
    text = re.sub(r'^\s*\d{1,3}\s*$', '', text, flags=re.MULTILINE)        # lone page numbers
    text = re.sub(r'^[\s\-_=|•·.★✓✗▪▸►>+*~]{1,}$', '', text, flags=re.MULTILINE)  # decorative lines
    text = re.sub(r'[\uf000-\uf8ff]', '', text)                             # PDF icon glyphs
    text = re.sub(r'\n{3,}', '\n\n', text)                                  # collapse blank lines
    lines = [l for l in text.split('\n') if len(l.strip()) > 2 or l.strip() == '']
    lines = [re.sub(r'[ \t]{2,}', ' ', l) for l in lines]
    return '\n'.join(lines).strip()

cv_text_cleaned = clean_cv_text(cv_text)   # ← one new line
print(cv_text_cleaned)               # verify it looks clean

Med Aziz BenTourkia
Business Information Systems Student
Diplômes et Formations
Licence en Informatique de Gestion ― Institue Supérieur de Gestion Tunis
Depuis septembre 2020
1ére année avec 15.73 Moyenne
2éme année avec 14.5
Baccalauréat en Sciences Techniques ― Lycée Grombalia Grombalia,Nabeul
De 2016 à 2020
Mention Très Bien
Expériences professionnelles
Stagiaire ― International Information Developments - IID 43 Av. Kheireddine Pacha, Tunis
De février 2023 à mai 2023
Développement et teste dans une application "Outil de communication"
Stagiaire ― Internation Information Developments - IID 43 Av. Kheireddine Pacha, Tunis
De juillet 2022 à août 2022
Développement et teste dans une application "modification de masse des barèmes"
Projets Académiques
Application des Remboursements Réalisation d'une application du remboursement social
pour une société avec C# .Net Framework avec SQL Server
Jeu de PENDU et MOTUS Réalisation de deux jeux de Pendu et Motus avec le langage C
Application de QU

In [43]:
def rag_retrieve_relevant_jobs(cv_text: str, top_k: int = 5) -> List[Dict]:
    # Step 1: embed the CV as a search query
    cv_emb = cohere_embed([cv_text_cleaned], input_type="search_query")
    faiss.normalize_L2(cv_emb)

    # Step 2: fetch 3x more candidates than needed for reranking
    scores, indices = index.search(cv_emb, top_k * 3)

    candidates = []
    candidate_docs = []
    for idx in indices[0]:
        job = scraped_jobs[idx].copy()
        candidates.append(job)
        candidate_docs.append(
            f"Title: {job['title']}\nCompany: {job['company']}\n{job['description'][:600]}"
        )

    # Step 3: Cohere rerank — reads CV and each job together (cross-encoder)
    rerank_results = co.rerank(
        query=cv_text_cleaned,
        documents=candidate_docs,
        model="rerank-multilingual-v3.0",
        top_n=top_k,
    )

    retrieved = []
    for r in rerank_results.results:
        job = candidates[r.index].copy()
        job["relevance_score"] = r.relevance_score
        retrieved.append(job)

    return retrieved

In [44]:
# Quick test
test_cv = cv_text
test_ret = rag_retrieve_relevant_jobs(test_cv, top_k=5)
print("Retrieved test jobs:")
for j in test_ret:
    print(f"  - {j['title']} || Description : {j['description'][:50]} || at {j['company']} (score: {j['relevance_score']:.3f})")

Retrieved test jobs:
  - Python Software Engineer || Description : Software Engineer Python with Angular, TypeScript
 || at Atem Corp (score: 0.947)
  - Senior Full-Stack & DevOps Engineer || Description : About Truss We started by building a B2B payment p || at Unknown (score: 0.880)
  - Full-Stack Developer || Description : We are looking for a strong Full-Stack Developer t || at Unknown (score: 0.859)
  - Senior Software Engineer – Java/Python || Description : ABOUT FANDUEL FanDuel Group is the premier mobile  || at FanDuel (score: 0.858)
  - Devops || Description : Headquarters: Buenos Aires Join Our Data Products  || at Mutt Data (score: 0.856)


## LLM Ranking Function (RAG Generation)

In [45]:
def llm_rank_retrieved_jobs(cv_text: str, retrieved_jobs: List[Dict]) -> str:
    if not retrieved_jobs:
        return "No jobs to rank." 
 
    jobs_text = ""
    for i, job in enumerate(retrieved_jobs[:5], 1):
        # NOTE: similarity score intentionally excluded
        jobs_text += (
            f"JOB {i}:\n"
            f"Title: {job['title']}\n"
            f"Company: {job['company']}\n"
            f"Description: {job['description'][:500]}\n"
            f"---\n"
        )
 
    messages = [
        {"role": "system", "content": (
            "You are an honest career coach in the tech field.\n"
            "Rank the following jobs by match to the CV based on YOUR OWN analysis "
            "of skills, experience level, and role fit.\n"
            "You MUST rank ALL jobs provided.\n"
            "For each job output EXACTLY this format:\n\n"
            "Rank #1\n"
            "Title: <job title>\n"
            "Company: <company name>\n"
            "Match Score: <your own score 0-100>\n"
            "Why: <2 sentence reason>\n"
            "-----"
        )},
        {"role": "user", "content": (
            f"CV:\n{cv_text[:1500]}\n\n"
            f"JOBS:\n{jobs_text}\n\n"
            f"Rank ALL {min(5, len(retrieved_jobs))} jobs from best to worst match."
        )}
    ]
 
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    outputs = model.generate(    
        **inputs,
        max_new_tokens=1024,
        do_sample=True,       
        temperature=0.1,
        top_p=0.9
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
 
    # Extract only assistant response
    for marker in ["<|im_start|>assistant", "<|assistant|>"]:
        if marker in response:
            response = response.split(marker)[-1].strip()
            break
 
    return response
 
 
# Test ranking
ranked = llm_rank_retrieved_jobs(test_cv, test_ret)
print(ranked)
 

system
You are an honest career coach in the tech field.
Rank the following jobs by match to the CV based on YOUR OWN analysis of skills, experience level, and role fit.
You MUST rank ALL jobs provided.
For each job output EXACTLY this format:

Rank #1
Title: <job title>
Company: <company name>
Match Score: <your own score 0-100>
Why: <2 sentence reason>
-----
user
CV:
Med Aziz BenTourkia
Business Information Systems Student
Diplômes et Formations
Licence en Informatique de Gestion ― Institue Supérieur de Gestion Tunis
Depuis septembre 2020
1ére année avec 15.73 Moyenne
2éme année avec 14.5
Baccalauréat en Sciences Techniques ― Lycée Grombalia Grombalia,Nabeul
De 2016 à 2020
Mention Très Bien
Expériences professionnelles
Stagiaire ― International Information Developments - IID 43 Av. Kheireddine Pacha, Tunis
De février 2023 à mai 2023
Développement et teste dans une application "Outil de communication"
Stagiaire ― Internation Information Developments - IID 43 Av. Kheireddine Pacha, Tun

## Gradio Chatbot

In [46]:
from transformers import TextIteratorStreamer
from threading import Thread

current_cv_text = ""
cached_retrieved = None
HARDCODED_CV = cv_text

def initialize_cv():
    global current_cv_text, cached_retrieved
    current_cv_text = HARDCODED_CV.strip()
    cached_retrieved = rag_retrieve_relevant_jobs(current_cv_text, top_k=5)
    print(f"✅ CV loaded: {len(current_cv_text)} characters")
    print(f"📊 Top matching jobs:")
    for i, job in enumerate(cached_retrieved[:3], 1):
        print(f"   {i}. {job['title']} at {job['company']} (match: {job['relevance_score']:.2f})")

def respond(message, history):
    global current_cv_text, cached_retrieved

    j1, j2, j3 = cached_retrieved[0], cached_retrieved[1], cached_retrieved[2]

    sys_prompt = f"""You are a career coach specialized in tech. You have access to the candidate's CV and their top matching job listings.
If the user asks about ANYTHING else (weather, sports, news, general knowledge, coding tutorials, etc.),
respond with exactly: "I can only help with questions about your CV and matched jobs."
Do not explain. Do not apologize. Just return that one sentence.

## Your behavior
- Answer ONLY based on what is in the CV and the job listings below
- Always tie your answer to a specific job title or a specific CV experience
- If the user asks something you cannot answer from the CV or jobs, say: "I don't have enough information in your CV to answer that"
- Never invent skills, metrics, or experiences
- Never give generic career advice that isn't grounded in the CV or jobs below
- Keep answers under 150 words
- Use bullet points only when listing 3+ items

## When the user asks about fit / usefulness / match for a role
1. Name the specific matching job(s) from the list below
2. List CV experiences that directly map to that job's requirements
3. List what is missing from the CV for that job, labeled clearly as "Gaps:"

## When the user asks about skill gaps
1. Compare CV skills against each job's requirements
2. List only gaps that appear in at least one job below
3. Do NOT suggest random skills not required by the jobs below

## When the user asks for interview prep
1. Base questions strictly on the job requirements below
2. Maximum 5 questions
3. For each question, cite which job it comes from

## When the user asks about CV improvement
1. List only 3 specific improvements based on the gap between your CV and the job requirements below
2. Each improvement must reference a specific job requirement from the list below
3. Format exactly as:
   - Improvement 1: <what to add/change> → needed for <job title>
   - Improvement 2: <what to add/change> → needed for <job title>
   - Improvement 3: <what to add/change> → needed for <job title>
4. Do NOT suggest general advice unless it is explicitly required by one of the jobs below

## CV
{current_cv_text[:1500]}

## Top Matching Jobs
Job 1: {j1['title']} at {j1['company']}
Requirements snapshot: {j1['description'][:200]}

Job 2: {j2['title']} at {j2['company']}
Requirements snapshot: {j2['description'][:200]}

Job 3: {j3['title']} at {j3['company']}
Requirements snapshot: {j3['description'][:200]}

## Important
The user is asking about THEIR OWN profile. Always say "your CV", "your experience", never "the candidate".
"""

    # Build conversation history — LLM memory is simulated by resending all prior turns
    messages = [{"role": "system", "content": sys_prompt}]
    for msg_dict in history:
        messages.append({"role": msg_dict["role"], "content": msg_dict["content"]})
    messages.append({"role": "user", "content": message})

    # Format into Qwen's expected chat format
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Truncate if conversation exceeds Qwen 3B's 2048-token context window
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)

    # Streamer pushes tokens to UI as they're generated
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    generation_kwargs = dict(
        **inputs,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.5,
        top_p=0.9,
        repetition_penalty=1.3,
        streamer=streamer,
    )

    # Generate in background thread so main thread can stream tokens to Gradio
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    # Stream tokens — history is a flat list of role/content dicts (type="messages" format)
    new_history = history + [{"role": "user", "content": message}, {"role": "assistant", "content": ""}]
    for token in streamer:
        new_history[-1]["content"] += token
        yield new_history, ""

    thread.join()


# ── Gradio UI ──────────────────────────────────────────────────────────────────
initialize_cv()

with gr.Blocks(theme=gr.themes.Soft(), title="Career Coach") as demo:
    gr.Markdown("# 🤖 AI Career Coach — Med Aziz BenTourkia")
    gr.Markdown("Ask about job matching, skill gaps, CV improvement, or career advice.")

    with gr.Row():
        with gr.Column(scale=1):
            cv_status = gr.Textbox(
                label="Loaded CV & Top Jobs",
                lines=6,
                interactive=False,
                value="\n".join([
                    f"{i+1}. {j['title']} at {j['company']}"
                    for i, j in enumerate(cached_retrieved[:3])
                ]) if cached_retrieved else "Initializing..."
            )

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(
                height=450,
                label="Career Coach",
                type="messages",
            )
            msg = gr.Textbox(
                placeholder="e.g. What jobs fit my profile? What skills am I missing?",
                label="Your question"
            )
            clear = gr.Button("Clear Conversation")

    msg.submit(respond, [msg, chatbot], [chatbot, msg], queue=True)
    clear.click(lambda: ([], ""), outputs=[chatbot, msg])

demo.queue()
demo.launch(share=True, debug=False)

✅ CV loaded: 2567 characters
📊 Top matching jobs:
   1. Python Software Engineer at Atem Corp (match: 0.95)
   2. Senior Full-Stack & DevOps Engineer at Unknown (match: 0.88)
   3. Full-Stack Developer at Unknown (match: 0.86)


/tmp/ipykernel_57/2820642321.py:118: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Career Coach") as demo:
/tmp/ipykernel_57/2820642321.py:135: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://e04576410fb95e227d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
